<a href="https://colab.research.google.com/github/Rishikesh-Mate/RealTime-Healthcare-Data-Pipeline/blob/main/src/producer/Healthcare_Data_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install confluent-kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 41.2 MB/s eta 0:00:00


In [ ]:
import random
import time
from datetime import datetime

# This function creates ONE fake medical record
def generate_patient_data():
    patient_ids = ["P-101", "P-102", "P-103", "P-104", "P-105"]

    record = {
        "patient_id": random.choice(patient_ids),
        "heart_rate": random.randint(60, 110),      # Normal to slightly high
        "spo2": random.randint(90, 100),            # Blood oxygen level
        "temperature": round(random.uniform(97.0, 103.0), 1),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    return record

# This loop runs the simulator
print("Starting Patient Vitals Stream... (Press 'Stop' button to end)")

while True:
    # Simulate a "batch" of 5 patients every 3 seconds
    for _ in range(5):
        data = generate_patient_data()
        print(data)

    print("-" * 30) # A separator to show the "burst"
    time.sleep(3)

Starting Patient Vitals Stream... (Press 'Stop' button to end)
{'patient_id': 'P-104', 'heart_rate': 108, 'spo2': 99, 'temperature': 97.7, 'timestamp': '2026-03-09 04:51:36'}
{'patient_id': 'P-101', 'heart_rate': 89, 'spo2': 99, 'temperature': 99.9, 'timestamp': '2026-03-09 04:51:36'}
{'patient_id': 'P-102', 'heart_rate': 101, 'spo2': 92, 'temperature': 97.1, 'timestamp': '2026-03-09 04:51:36'}
{'patient_id': 'P-101', 'heart_rate': 62, 'spo2': 91, 'temperature': 102.6, 'timestamp': '2026-03-09 04:51:36'}
{'patient_id': 'P-103', 'heart_rate': 84, 'spo2': 96, 'temperature': 102.8, 'timestamp': '2026-03-09 04:51:36'}
------------------------------
{'patient_id': 'P-102', 'heart_rate': 99, 'spo2': 99, 'temperature': 99.2, 'timestamp': '2026-03-09 04:51:39'}
{'patient_id': 'P-105', 'heart_rate': 74, 'spo2': 92, 'temperature': 98.3, 'timestamp': '2026-03-09 04:51:39'}
{'patient_id': 'P-102', 'heart_rate': 96, 'spo2': 90, 'temperature': 101.4, 'timestamp': '2026-03-09 04:51:39'}
{'patient_id'

KeyboardInterrupt: 

In [1]:
import random
import time
import json
from datetime import datetime

# This function creates ONE fake medical record
def generate_patient_data():
    patient_ids = ["P-101", "P-102", "P-103", "P-104", "P-105"]

    # Base "Clean" Data
    record = {
        "patient_id": random.choice(patient_ids),
        "heart_rate": random.randint(60, 110),      # Normal to slightly high
        "spo2": random.randint(90, 100),            # Blood oxygen level
        "temperature": round(random.uniform(97.0, 103.0), 1),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    # --- Error Injection Logic (10% chance of "dirty" data) ---
    if random.random() < 0.10:
        error_type = random.choice(['out_of_range', 'missing_value', 'bad_type'])

        if error_type == 'out_of_range':
            record["heart_rate"] = 999  # Impossible heart rate
        elif error_type == 'missing_value':
            record["spo2"] = None       # Simulates sensor failure
        elif error_type == 'bad_type':
            record["temperature"] = "N/A" # Simulates a formatting error

    return record

# This loop runs the simulator
print("Starting Patient Vitals Stream... (Press 'Stop' button to end)")

while True:
    # Simulate a "batch" of 5 patients every 3 seconds
    for _ in range(5):
        data = generate_patient_data()

        # Convert Python dictionary to a JSON string
        json_data = json.dumps(data)
        print(json_data)

    print("-" * 30) # A separator to show the "burst"
    time.sleep(3)

Starting Patient Vitals Stream... (Press 'Stop' button to end)
{"patient_id": "P-104", "heart_rate": 76, "spo2": 90, "temperature": 101.3, "timestamp": "2026-03-25 06:05:47"}
{"patient_id": "P-103", "heart_rate": 81, "spo2": 92, "temperature": 97.4, "timestamp": "2026-03-25 06:05:47"}
{"patient_id": "P-105", "heart_rate": 103, "spo2": 93, "temperature": 101.6, "timestamp": "2026-03-25 06:05:47"}
{"patient_id": "P-101", "heart_rate": 76, "spo2": 93, "temperature": 99.5, "timestamp": "2026-03-25 06:05:47"}
{"patient_id": "P-105", "heart_rate": 78, "spo2": null, "temperature": 102.8, "timestamp": "2026-03-25 06:05:47"}
------------------------------
{"patient_id": "P-102", "heart_rate": 74, "spo2": null, "temperature": 102.3, "timestamp": "2026-03-25 06:05:50"}
{"patient_id": "P-105", "heart_rate": 87, "spo2": 95, "temperature": 98.9, "timestamp": "2026-03-25 06:05:50"}
{"patient_id": "P-104", "heart_rate": 69, "spo2": 94, "temperature": 103.0, "timestamp": "2026-03-25 06:05:50"}
{"patien

KeyboardInterrupt: 

In [21]:
import os
import json
import time
import random
from datetime import datetime
from confluent_kafka import Producer

# 1. Credentials Setup
service_uri = "kafka-114b7e84-rishikeshmate09-5283.l.aivencloud.com:21915"

conf = {
    'bootstrap.servers': service_uri,
    'security.protocol': 'SSL',
    'ssl.ca.location': 'ca.pem',
    'ssl.certificate.location': 'service.cert',
    'ssl.key.location': 'service.key',
}

# 2. Create the Producer (No more PEM lib errors!)
try:
    producer = Producer(conf)
    print("🚀 SUCCESS! Connected to the cloud. Starting stream...")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    raise

# 3. Stream Data
topic_name = 'patient-vitals'
try:
    while True:
        data = {
            "patient_id": random.choice(["P-101", "P-102", "P-103"]),
            "heart_rate": random.randint(60, 110),
            "spo2": random.randint(92, 100),
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        producer.produce(topic_name, value=json.dumps(data).encode('utf-8'))
        print(f"✅ Sent Record for {data['patient_id']} | HR: {data['heart_rate']}")
        producer.flush()
        time.sleep(3)
except KeyboardInterrupt:
    print("\n🛑 Stopped.")

🚀 SUCCESS! Connected to the cloud. Starting stream...
✅ Sent Record for P-101 | HR: 83
✅ Sent Record for P-101 | HR: 64
✅ Sent Record for P-101 | HR: 61
✅ Sent Record for P-103 | HR: 82
✅ Sent Record for P-103 | HR: 85
✅ Sent Record for P-101 | HR: 75
✅ Sent Record for P-101 | HR: 110
✅ Sent Record for P-103 | HR: 100
✅ Sent Record for P-102 | HR: 96
✅ Sent Record for P-103 | HR: 110
✅ Sent Record for P-103 | HR: 110
✅ Sent Record for P-102 | HR: 107
✅ Sent Record for P-101 | HR: 110
✅ Sent Record for P-103 | HR: 86
✅ Sent Record for P-101 | HR: 103
✅ Sent Record for P-102 | HR: 68
✅ Sent Record for P-101 | HR: 91
✅ Sent Record for P-101 | HR: 78
✅ Sent Record for P-101 | HR: 85
✅ Sent Record for P-101 | HR: 70
✅ Sent Record for P-101 | HR: 105
✅ Sent Record for P-103 | HR: 80
✅ Sent Record for P-102 | HR: 84
✅ Sent Record for P-103 | HR: 61
✅ Sent Record for P-102 | HR: 67
✅ Sent Record for P-102 | HR: 95
✅ Sent Record for P-101 | HR: 96
✅ Sent Record for P-103 | HR: 85
✅ Sent Record 